# DNA Fiber Assay Analysis of ORC1 mutants


**Purpose:** Compare replication speed and inter-origin distance between WT and MGS variants.

**Author:** Elena Lopatukhina
**Date:** 2026-08-17

## Workflow
1. Parameters import
2. Data loading
3. Data propcessing
4. Basic statistics calculation
5. Processing outliers
6. Statistical analysis
7. Tables export
8. Graphs plot

# 1. Parameters

In [1]:
INPUT_DIR = "/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZED"
OUTPUT_DIR = "/mnt/c/users/helen/Desktop/290925_GOOD/RESULTS"

pixel_size = 0.16125 # µm
conversion_factor = 2.59 # kb/µm
time = 20 # minutes

# 2. Import data

## 2.1 Data loading

In [2]:
from utils import load_data, data_subset

data = load_data(dir = INPUT_DIR, pixel_size=pixel_size)

print(f"Total number of measurements for analysis is: {data.shape[0]}")

Total number of measurements for analysis is: 1280


In [6]:
data

,Sample_name,File,Measurement_type,Length,ROI,Path
0,MGS1,MGS1_5-02_Fiber_length,Fiber_length,8.249066,0576-0284,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
1,MGS1,MGS1_5-02_Fiber_length,Fiber_length,6.534656,0582-0336,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
2,MGS1,MGS1_5-02_Fiber_length,Fiber_length,3.562174,0541-0074,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
3,MGS1,MGS1_5-02_Fiber_length,Fiber_length,5.344470,0537-0044,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
4,MGS1,MGS1_5-02_Fiber_length,Fiber_length,4.150414,0615-0480,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
...,...,...,...,...,...,...
1275,WT,WT6-05_Interorigin_distance,Interorigin_distance,26.806845,0161-0369,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
1276,WT,WT6-05_Interorigin_distance,Interorigin_distance,17.924873,0145-0611,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
1277,WT,WT6-10_Interorigin_distance,Interorigin_distance,13.701251,0691-0712,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
1278,WT,WT6-10_Interorigin_distance,Interorigin_distance,22.469059,0699-0866,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...


In [3]:
# Sample name 29.09.25
data['Sample_name'] = data['File'].apply(lambda x: x.split('_')[1] if 'siORC1_MGS' in x else x.split('-')[0])
data['Sample_name'] = data['Sample_name'].apply(lambda x: 'WT' if 'WT' in x else x)
data['Sample_name'] = data['Sample_name'].apply(lambda x: 'siSCR' if 'siSCR' in x else x)
data['Sample_name'] = data['Sample_name'].apply(lambda x: x.split('-')[0] if 'MGS' in x else x)
data['Sample_name'] = data['Sample_name'].apply(lambda x: 'siORC1' if 'siORC1' in x else x)
data['Sample_name'] = data['Sample_name'].apply(lambda x: x.split('_')[0] if 'MGS' in x else x)

In [4]:
# Check sample names
data['Sample_name'].unique()

<StringArray>
['MGS1', 'MGS2', 'MGS5', 'siORC1', 'MGS3', 'MGS4', 'WT', 'siSCR']
Length: 8, dtype: str

### 2.3 Split data into replication speed and IOD dataframe

In [7]:
speed = data_subset(df = data, measurement_type = 'Fiber_length')
print(f"The total amount of fibers measurements is: {speed.shape[0]}")

iod = data_subset(df = data, measurement_type = 'Interorigin_distance')
print(f"The total amount of IOD measurements is: {iod.shape[0]}")

The total amount of fibers measurements is: 789
The total amount of IOD measurements is: 491


### 2.4 Checking the number of measurements for each sample

#### 2.4.1 Replication speed
Divide this number by 2 because these are green and red tracks separately.

In [8]:
speed.groupby('Sample_name')['File'].count()

Sample_name
MGS1       98
MGS2       67
MGS3      106
MGS4       71
MGS5       38
WT        144
siORC1    149
siSCR     116
Name: File, dtype: int64

#### 2.4.2. IOD

In [9]:
iod.groupby('Sample_name')['File'].count()

Sample_name
MGS1       20
MGS2       39
MGS3       34
MGS4       35
MGS5       40
WT        120
siORC1     93
siSCR     110
Name: File, dtype: int64

# 3. Data processing

In [10]:
from utils import speed_processing, iod_processing

replication_speed = speed_processing(df = speed, conversion_factor = conversion_factor, time = time)
iod_kb = iod_processing(df = iod, conversion_factor = conversion_factor)

The following files contain an odd number of fibers will be removed:
WT5-12_Fiber_length
siORC1-06_Fiber_length
siORC1_MGS2-10_Fiber_length
siORC1_MGS4-15_Fiber_length
siORC1_WT-13_Fiber_length


In [11]:
replication_speed.head()

,Sample_name,File,Speed_kb_min,ROI,Path
0,MGS1,MGS1_5-02_Fiber_length,1.914492,"[0576-0284, 0582-0336]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
1,MGS1,MGS1_5-02_Fiber_length,1.153410,"[0541-0074, 0537-0044]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
2,MGS1,MGS1_5-02_Fiber_length,1.854352,"[0615-0480, 0603-0429]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
3,MGS1,MGS1_5-02_Fiber_length,1.238400,"[0568-0298, 0567-0264]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
4,MGS2,MGS2_2-02_Fiber_length,1.559062,"[0059-0813, 0069-0858]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...


In [12]:
iod_kb.head()

,Sample_name,File,IOD_kb,ROI,Path
8,MGS1,MGS1_5-02_Interorigin_distance,86.108500,0693-0558,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
15,MGS2,MGS2_2_Interorigin_distance,42.967799,0749-0104,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
16,MGS2,MGS2_2_Interorigin_distance,40.690422,0814-0290,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
17,MGS2,MGS2_2_Interorigin_distance,48.060471,0260-0266,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
18,MGS2,MGS2_2_Interorigin_distance,50.506573,0423-0572,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...


# 4. Basic statistics calculation

In [13]:
from utils import description_stats

stats_speed = description_stats(replication_speed, col = "Speed_kb_min")
stats_iod = description_stats(iod_kb, col = "IOD_kb")

In [14]:
stats_speed

,Count,Mean,Median,SD
Sample_name,,,,
MGS1,49,1.806583,1.729416,0.525473
MGS2,27,1.856669,1.704086,0.496043
MGS3,53,1.663987,1.562069,0.493414
MGS4,35,1.945943,1.896617,0.615471
MGS5,19,1.826482,1.933056,0.733991
WT,71,1.418733,1.423643,0.318855
siORC1,74,1.651082,1.515920,0.598529
siSCR,58,1.376885,1.349982,0.445264


In [15]:
stats_iod

,Count,Mean,Median,SD
Sample_name,,,,
MGS1,20,49.743550,46.053931,17.522491
MGS2,39,43.949451,44.531016,11.988720
MGS3,34,47.599608,44.874105,13.941493
MGS4,35,62.626306,62.018751,18.861275
MGS5,40,63.725698,60.093651,25.581089
WT,120,41.212308,40.671628,15.052449
siORC1,93,46.321334,45.044710,15.932851
siSCR,110,38.673214,35.308954,13.847129


# 5. Processing outliers

In [16]:
from utils import outliers

speed_outliers = outliers(df = replication_speed, col = 'Speed_kb_min')
iod_outliers = outliers(df = iod_kb, col = 'IOD_kb')

In [17]:
speed_outliers

,Sample_name,File,Speed_kb_min,ROI,Path
13,MGS2,MGS2_6-16_Fiber_length,2.962157,"[0341-0405, 0311-0335]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
17,MGS5,MGS5_5-11_Fiber_length,3.516257,"[0747-0259, 0810-0193]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
18,MGS5,MGS5_5-11_Fiber_length,2.939813,"[0894-0474, 0946-0417]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
90,siORC1,siORC1-14_Fiber_length,3.063288,"[0391-0887, 0387-0801]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
92,siORC1,siORC1-16_Fiber_length,3.220591,"[0632-1239, 0621-1152]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
118,siORC1,siORC1_6-09_Fiber_length,3.453946,"[0335-1151, 0370-1075]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
153,MGS1,siORC1_MGS1-20_Fiber_length,3.346550,"[0877-1008, 0935-0958]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
184,MGS1,siORC1_MGS1-36_Fiber_length,3.957241,"[0765-0175, 0787-0081]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
256,MGS4,siORC1_MGS4-05_Fiber_length,3.192338,"[0143-0700, 0212-0655]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
268,MGS4,siORC1_MGS4-14_Fiber_length,2.964120,"[0244-1100, 0283-1032]",/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...


In [18]:
iod_outliers

,Sample_name,File,IOD_kb,ROI,Path
58,MGS5,MGS5_5-07_Interorigin_distance,113.733132,0341-0823,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
60,MGS5,MGS5_5-07_Interorigin_distance,121.273995,0649-0368,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
63,MGS5,MGS5_5-08_Interorigin_distance,91.213283,0578-0391,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
64,MGS5,MGS5_5-09_Interorigin_distance,108.715635,0505-0256,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
112,MGS5,MGS5_5-18_Interorigin_distance,121.031348,0452-0458,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
114,MGS5,MGS5_5-25_Interorigin_distance,92.926850,0574-1106,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
115,MGS5,MGS5_5-26_Interorigin_distance,90.447753,0764-0284,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
271,siORC1,siORC1_5-06_Interorigin_distance,103.518972,0621-0530,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
273,siORC1,siORC1_5-08_Interorigin_distance,88.952193,0532-0412,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...
684,MGS4,siORC1_MGS4_Interorigin_distance,96.211986,0581-0619,/mnt/c/users/helen/Desktop/290925_GOOD/ANALYZE...


In [ ]:
# Delete ouliers from the dataframes
speed_clean = replication_speed.drop(index=speed_outliers.index)
iod_kb_clean = iod_kb.drop(index=iod_outliers.index)

# 6. Statistical analysis
Mann-Whetney test (U-test)